In [1]:
import requests
import pandas as pd
from bs4 import BeautifulSoup

In [2]:
# link_squad = 'startseite'
# url_squad = f'https://www.transfermarkt.com.br/{self.league}/{link_type}/wettbewerb/GB1/plus/?saison_id={n_season}'

# link_goals = 'spieltag'
# url_goals = f'https://www.transfermarkt.com.br/{self.league}/{link_type}/wettbewerb/GB1/saison_id/{n_season}/spieltag/{n_round}'

# link_match = 'spieltag'
# url_match = f'https://www.transfermarkt.com.br/{self.league}/{link_type}/wettbewerb/GB1/saison_id/{n_season}/spieltag/{n_round}'

# link_table = 'tabelle'
# url_table = f'https://www.transfermarkt.com.br/{self.league}/{link_type}/wettbewerb/GB1/saison_id/{n_season}'

# link_title = 'erfolge'
# url_title = f'https://www.transfermarkt.com.br/{self.league}/{link_type}/wettbewerb/GB1'

# link_place = 'spieltagtabelle'
# url_place = f'https://www.transfermarkt.com.br/{self.league}/{link_type}/wettbewerb/GB1/saison_id/{n_season}/spieltag/{n_round}'

# premier-league

In [3]:
headers = {
    "User-Agent":"Mozilla/5.0 (Windows NT 10.0; Win64; x64; rv:150.0) Gecko/20100101 Firefox/150.0"
}

## Functions

This project required the development of six specialized scraping functions responsible for collecting match results, match events, league standings, historical standings progression, titles, and squad information. The architecture was designed to be modular and reusable, enabling data extraction from both completed and ongoing seasons through round-specific queries. Furthermore, the same functions can be easily adapted to other competitions available on Transfermarkt, as long as they share the same underlying page structure. Each function and its role within the project are presented in the following sections.

In [ ]:
def get_events(headers, league, n_season, n_round):
        # Variables Needed
        goals_list = []
        count_event = 0
        n_match = 0
        season_id = f'PL-{n_season}'
        
        # Inicializing Beautiful Soup
        url = f'https://www.transfermarkt.com.br/{league}/spieltag/wettbewerb/GB1/saison_id/{n_season}/spieltag/{n_round}'
        response = requests.get(url, headers=headers)
        response.status_code
        soup = BeautifulSoup(response.content, "lxml")

        # Storing all match related data in a single list
        all_matches = soup.find_all('table', {'style':'border-top: 0 !important;'})
        
        
        for match in all_matches:
            # Creating match identifier
            n_match += 1
            match_id = f'M-{n_season}-{n_match:02d}'
            
            # Storing all events data in a single list
            event = match.find_all('tr', {'class':'no-border spieltagsansicht-aktionen'})

            # List with the entire class necessary to get the home and away team's names
            gross_h_team = match.find('td', {'class':'rechts hauptlink no-border-rechts hide-for-small spieltagsansicht-vereinsname'})
            gross_a_team = match.find('td', {'class':'hauptlink zentriert no-border-rechts no-border-links hide-for-small spieltagsansicht-wappen'})

            # Checking for a possible forum buttom
            home_forum_check = gross_h_team.find('a').get('href')
            away_forum_check = gross_a_team.find('a').get('href')

            # Different ways to get the title depending if it has the forum buttom
            if 'forum' in home_forum_check and 'forum' in away_forum_check:
                h_team = gross_h_team.find_all('a')[1].get('title')
                a_team = gross_a_team.find_all('a')[1].get('title')
            elif 'forum' in home_forum_check:
                h_team = gross_h_team.find_all('a')[1].get('title')
                a_team = gross_a_team.find('a').get('title')
            elif 'forum' in away_forum_check:
                h_team = gross_h_team.find('a').get('title')
                a_team = gross_a_team.find_all('a')[1].get('title')
            else:
                h_team = gross_h_team.find('a').get('title')
                a_team = gross_a_team.find('a').get('title')

            # Access one by one all match related events
            for row in event:
                # Temporary list to store events of a single match
                temp = []
                temp.append(season_id)
                temp.append(match_id)

                # Creating event identifier
                count_event += 1
                event_id = f"E-{n_season}-{n_round}-{count_event:04d}"
                temp.append(event_id)

                # Transfermarkt separates home and away team events
                # Home Team Events
                try: 
                    event_type = row.find('td', {'class':'rechts no-border-rechts spieltagsansicht'}).find_all('span')[2].get('class')[1]
                    event_minute = row.find('td', {'class':'zentriert no-border-links'}).string
                    temp.append(h_team)
                    temp.append(event_minute)
                
                # Away Team Events
                except: 
                    event_type = row.find('td', {'class':'links no-border-links spieltagsansicht'}).find('span').get('class')[1]
                    event_minute = row.find('td', {'class':'zentriert no-border-rechts'}).string
                    temp.append(a_team)
                    temp.append(event_minute)

                # Event Type Information
                if event_type == 'icon-tor-formation': temp.append(0) # Normal Goal
                elif event_type == 'icon-elfmeter-formation': temp.append(1) # Penalty Goal
                elif event_type == 'icon-eigentor-formation': temp.append(2) # Own Goal
                elif event_type == 'icon-verschossener-elfmeter-formation': temp.append(-2) # Penalty Missed
                else: temp.append(-1) # Red Cards

                # Player wich made the action
                player = row.find('a').get('title')
                temp.append(player)    

                # Inserting all events related to the match into the list
                goals_list.append(temp)

        goals_list.insert(0,['season_id', 'match_id', 'event_id','goal_score_team','goal_minute','goal_type', 'goal_scorer_name'])
        return goals_list

In [5]:
events = []
head = []

for season in range(1992,2001):
    if season > 1994: season_round = 39
    else: season_round = 43

    for round in range(1,season_round+1):
        temp = get_events(headers,'premier-league',season,round)
        head=temp[0]
        events += temp[1:]
events.insert(0,head)

df_events = pd.DataFrame(events[1:], columns=events[0])
display(df_events)

KeyboardInterrupt: 